
This Databricks notebook, titled **understanding_quick_comm**, is designed to perform a comprehensive end-to-end data engineering and analytics workflow on an e-commerce dataset (Olist).

The notebook is organized into several key phases:

- **Data Ingestion & Integration:** It loads multiple CSV files (orders, items, payments, reviews, etc.) using pandas and merges them into a single Master Dataset (df) for unified analysis.
- **Data Transformation:** It handles schema cleaning, such as converting timestamp strings into datetime objects and translating product category names from Portuguese to English.
- **Feature Engineering:** It calculates business-specific metrics, such as delivery_delay_days, and categorizes orders as "Delayed" or "On Time."
- **Data Persistence:** It converts the processed Pandas DataFrame into a Spark DataFrame to save it as a permanent Delta table (final_quick_comm_dataset) in the Databricks workspace catalog.
- **Business Intelligence:** The final sections generate high-level Executive KPIs (GMV, AOV, Repeat Purchase Rate) and analyze the correlation between delivery performance and customer satisfaction scores.

# Load the Files First in DataFrames

In [0]:
import pandas as pd
import numpy as np

# Load datasets
orders = pd.read_csv("quick_comm_tables/olist_orders_dataset.csv")
items = pd.read_csv("quick_comm_tables/olist_order_items_dataset.csv")
payments = pd.read_csv("quick_comm_tables/olist_order_payments_dataset.csv")
reviews = pd.read_csv("quick_comm_tables/olist_order_reviews_dataset.csv")
products = pd.read_csv("quick_comm_tables/olist_products_dataset.csv")
customers = pd.read_csv("quick_comm_tables/olist_customers_dataset.csv")
category = pd.read_csv("quick_comm_tables/product_category_name_translation.csv")

## Translate Product Categories

In [0]:
products = products.merge(
    category,
    on="product_category_name",
    how="left"
)

## Build MASTER DATASET

In [0]:
df = orders.merge(items, on="order_id", how="left") \
           .merge(payments, on="order_id", how="left") \
           .merge(reviews, on="order_id", how="left") \
           .merge(products, on="product_id", how="left") \
           .merge(customers, on="customer_id", how="left")

In [0]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "review_creation_date",
    "review_answer_timestamp"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

In [0]:
%sql
SELECT * FROM workspace.default.final_quick_comm_dataset LIMIT 100

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,delivery_delay_days,delivery_status
17ae98e755b32ae8ec73470952de4de2,80d88401a15807dd28990dd932495bc6,delivered,2017-11-16T23:07:13.000Z,2017-11-16T23:16:41.000Z,2017-11-23T22:13:39.000Z,2017-12-06T18:44:26.000Z,2017-12-07T00:00:00.000Z,1.0,74fd207bb446a56583153f85de68b73c,54965bbe3e4f07ae045b90b0b8541f52,2017-11-22T23:16:41.000Z,120.0,21.33,2.0,voucher,1.0,75.0,d83ab20963957eab4af21ea4f71f16e0,5.0,null,null,2017-12-09T00:00:00.000Z,2017-12-12T02:33:05.000Z,cama_mesa_banho,41.0,729.0,1.0,1950.0,20.0,30.0,30.0,bed_bath_table,0fef599ec418d6a774d937017ee7ec2c,8820,mogi das cruzes,SP,-1.0,On Time
17ae98e755b32ae8ec73470952de4de2,80d88401a15807dd28990dd932495bc6,delivered,2017-11-16T23:07:13.000Z,2017-11-16T23:16:41.000Z,2017-11-23T22:13:39.000Z,2017-12-06T18:44:26.000Z,2017-12-07T00:00:00.000Z,1.0,74fd207bb446a56583153f85de68b73c,54965bbe3e4f07ae045b90b0b8541f52,2017-11-22T23:16:41.000Z,120.0,21.33,1.0,credit_card,2.0,66.33,d83ab20963957eab4af21ea4f71f16e0,5.0,null,null,2017-12-09T00:00:00.000Z,2017-12-12T02:33:05.000Z,cama_mesa_banho,41.0,729.0,1.0,1950.0,20.0,30.0,30.0,bed_bath_table,0fef599ec418d6a774d937017ee7ec2c,8820,mogi das cruzes,SP,-1.0,On Time
9a06f6b071c10b11ce8e6e8646dae224,89610c63a51a5b2ee28d3b69c6bfff3c,delivered,2018-06-09T09:22:08.000Z,2018-06-09T09:37:14.000Z,2018-06-11T14:02:00.000Z,2018-06-19T20:28:27.000Z,2018-07-16T00:00:00.000Z,1.0,d325b4b600e06bd11a1f8afdfdf3679c,06a2c3af7b3aee5d69171b0e14f0ee87,2018-06-19T09:32:12.000Z,154.99,30.11,1.0,credit_card,7.0,185.1,e9e81e73f44a0fc9212369c351f9d08b,4.0,otimo,bom,2018-06-20T00:00:00.000Z,2018-06-21T19:04:47.000Z,beleza_saude,46.0,1290.0,1.0,950.0,25.0,12.0,19.0,health_beauty,921c8a95cd7f1ba09d26223716253455,13070,campinas,SP,-27.0,On Time
d0028facea13f508e880202d7097a5a1,d2509c13692836fc0449e88cf9eb4858,delivered,2018-04-20T12:57:23.000Z,2018-04-25T03:51:13.000Z,2018-04-25T15:25:00.000Z,2018-04-27T12:08:59.000Z,2018-05-09T00:00:00.000Z,1.0,8cefe1c6f2304e7e6825150218ffc58c,ea8482cd71df3c1969d7b9473ff13abc,2018-05-02T03:51:13.000Z,27.99,7.39,1.0,boleto,1.0,35.38,ffc1a4080d67b1d3d06749f4f2d3ee59,4.0,null,null,2018-04-28T00:00:00.000Z,2018-05-01T20:41:51.000Z,telefonia,59.0,818.0,6.0,300.0,17.0,4.0,12.0,telephony,00050ab1314c0e55a6ca13cf7181fecf,13084,campinas,SP,-12.0,On Time
812114e24682287b1f3eb86f112707de,de94552cb07b7cbb5219d1616580a65c,delivered,2017-12-04T21:36:35.000Z,2017-12-05T09:30:45.000Z,2017-12-07T15:19:17.000Z,2017-12-19T15:53:56.000Z,2017-12-28T00:00:00.000Z,1.0,06bf70b6e1d67d96308235ef350edc61,2c9e548be18521d1c43cde1c582c6de8,2017-12-11T08:30:50.000Z,79.9,15.32,1.0,boleto,1.0,95.22,7a3fc44fbc09b8f38a5bead35c987785,4.0,null,null,2017-12-20T00:00:00.000Z,2017-12-20T23:27:27.000Z,brinquedos,48.0,284.0,2.0,950.0,105.0,10.0,11.0,toys,3526e8083b82706f0a66e58c84044cab,32676,betim,MG,-9.0,On Time
bc6c1d04178771a3ac9547f4770ab5f0,8fd50d32ac51d30d03b416bf7b27e72d,delivered,2017-10-27T10:27:03.000Z,2017-10-27T10:35:22.000Z,2017-10-31T21:53:55.000Z,2017-11-09T21:41:44.000Z,2017-11-17T00:00:00.000Z,1.0,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2017-11-03T10:35:22.000Z,59.9,17.67,1.0,credit_card,4.0,77.57,c123f6ddcc806bd9bed231d6163f267e,5.0,null,null,2017-11-10T00:00:00.000Z,2017-11-13T10:01:10.000Z,ferramentas_jardim,56.0,348.0,2.0,1550.0,30.0,22.0,30.0,garden_tools,8cf98f

In [0]:
df.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,shipping_limit_date,price,freight_value,payment_sequential,payment_installments,payment_value,review_score,review_creation_date,review_answer_timestamp,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,customer_zip_code_prefix
count,119143,118966,117057,115722,119143,118310.000000,118310,118310.000000,118310.000000,119140.000000,119140.000000,119140.000000,118146.000000,118146,118146,116601.000000,116601.000000,116601.000000,118290.000000,118290.000000,118290.000000,118290.000000,119143.000000
mean,2017-12-29 18:36:13.115760128,2017-12-30 04:49:18.425726976,2018-01-03 08:24:34.395524864,2018-01-12 20:55:38.199616,2018-01-22 15:21:10.241642240,1.196543,2018-01-05 22:06:13.308807424,120.646603,20.032387,1.094737,2.941246,172.735135,4.015582,2018-01-11 13:17:50.103092992,2018-01-14 17:00:35.769302784,48.767498,785.967822,2.205161,2112.250740,30.265145,16.619706,23.074799,35033.451298
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00,1.000000,2016-09-19 00:15:34,0.850000,0.000000,1.000000,0.000000,0.000000,1.000000,2016-10-02 00:00:00,2016-10-07 18:32:28,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000,1003.000000
25%,2017-09-10 20:15:46,2017-09-11 15:50:48.500000,2017-09-14 19:52:12,2017-09-22 21:54:31.249999872,2017-10-02 00:00:00,1.000000,2017-09-18 14:30:33,39.900000,13.080000,1.000000,1.000000,60.850000,4.000000,2017-09-22 00:00:00,2017-09-25 11:15:40.750000128,42.000000,346.000000,1.000000,300.000000,18.000000,8.000000,15.000000,11250.000000
50%,2018-01-17 11:59:12,2018-01-17 16:49:49,2018-01-23 17:03:08,2018-02-01 03:17:55,2018-02-14 00:00:00,1.000000,2018-01-25 04:11:15.500000,74.900000,16.280000,1.000000,2.000000,108.160000,5.000000,2018-02-01 00:00:00,2018-02-03 12:04:23,52.000000,600.000000,1.000000,700.000000,25.000000,13.000000,20.000000,24240.000000
75%,2018-05-03 13:18:30,2018-05-03 16:56:53,2018-05-07 14:57:00,2018-05-15 00:08:31.500000,2018-05-25 00:00:00,1.000000,2018-05-10 02:51:40.249999872,134.900000,21.180000,1.000000,4.000000,189.240000,5.000000,2018-05-15 00:00:00,2018-05-17 10:48:59,57.000000,983.000000,3.000000,1800.000000,38.000000,20.000000,30.000000,58475.000000
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00,21.000000,2020-04-09 22:35:08,6735.000000,409.680000,29.000000,24.000000,13664.080000,5.000000,2018-08-31 00:00:00,2018-10-29 12:27:35,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000,99990.000000
std,NaN,NaN,NaN,NaN,NaN,0.699489,NaN,184.109691,15.836850,0.730141,2.777848,267.776077,1.400436,NaN,NaN,10.033540,652.584121,1.717452,3786.695111,16.189367,13.453584,11.749139,29823.198969


In [0]:
df.isnull().sum()

order_id                              0
customer_id                           0
order_status                          0
order_purchase_timestamp              0
order_approved_at                   177
order_delivered_carrier_date       2086
order_delivered_customer_date      3421
order_estimated_delivery_date         0
order_item_id                       833
product_id                          833
seller_id                           833
shipping_limit_date                 833
price                               833
freight_value                       833
payment_sequential                    3
payment_type                          3
payment_installments                  3
payment_value                         3
review_id                           997
review_score                        997
review_comment_title             105154
review_comment_message            68898
review_creation_date                997
review_answer_timestamp             997
product_category_name              2542


## Create BUSINESS METRICS

In [0]:
#delivery delay
df["delivery_delay_days"] = (
    df["order_delivered_customer_date"] -
    df["order_estimated_delivery_date"]
).dt.days
#delivery status
df["delivery_status"] = np.where(
    df["delivery_delay_days"] > 0,
    "Delayed",
    "On Time"
)

# CREATE dataset with the above df

In [0]:
%sql
SELECT current_catalog() as catalog, current_schema() as schema

catalog,schema
workspace,default


In [0]:
# Save as a permanent Delta table with explicit catalog and schema
spark_df = spark.createDataFrame(df)
spark_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.final_quick_comm_dataset")

In [0]:
%sql
SELECT * FROM workspace.default.final_quick_comm_dataset LIMIT 100

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,delivery_delay_days,delivery_status
17ae98e755b32ae8ec73470952de4de2,80d88401a15807dd28990dd932495bc6,delivered,2017-11-16T23:07:13.000Z,2017-11-16T23:16:41.000Z,2017-11-23T22:13:39.000Z,2017-12-06T18:44:26.000Z,2017-12-07T00:00:00.000Z,1.0,74fd207bb446a56583153f85de68b73c,54965bbe3e4f07ae045b90b0b8541f52,2017-11-22T23:16:41.000Z,120.0,21.33,2.0,voucher,1.0,75.0,d83ab20963957eab4af21ea4f71f16e0,5.0,null,null,2017-12-09T00:00:00.000Z,2017-12-12T02:33:05.000Z,cama_mesa_banho,41.0,729.0,1.0,1950.0,20.0,30.0,30.0,bed_bath_table,0fef599ec418d6a774d937017ee7ec2c,8820,mogi das cruzes,SP,-1.0,On Time
17ae98e755b32ae8ec73470952de4de2,80d88401a15807dd28990dd932495bc6,delivered,2017-11-16T23:07:13.000Z,2017-11-16T23:16:41.000Z,2017-11-23T22:13:39.000Z,2017-12-06T18:44:26.000Z,2017-12-07T00:00:00.000Z,1.0,74fd207bb446a56583153f85de68b73c,54965bbe3e4f07ae045b90b0b8541f52,2017-11-22T23:16:41.000Z,120.0,21.33,1.0,credit_card,2.0,66.33,d83ab20963957eab4af21ea4f71f16e0,5.0,null,null,2017-12-09T00:00:00.000Z,2017-12-12T02:33:05.000Z,cama_mesa_banho,41.0,729.0,1.0,1950.0,20.0,30.0,30.0,bed_bath_table,0fef599ec418d6a774d937017ee7ec2c,8820,mogi das cruzes,SP,-1.0,On Time
9a06f6b071c10b11ce8e6e8646dae224,89610c63a51a5b2ee28d3b69c6bfff3c,delivered,2018-06-09T09:22:08.000Z,2018-06-09T09:37:14.000Z,2018-06-11T14:02:00.000Z,2018-06-19T20:28:27.000Z,2018-07-16T00:00:00.000Z,1.0,d325b4b600e06bd11a1f8afdfdf3679c,06a2c3af7b3aee5d69171b0e14f0ee87,2018-06-19T09:32:12.000Z,154.99,30.11,1.0,credit_card,7.0,185.1,e9e81e73f44a0fc9212369c351f9d08b,4.0,otimo,bom,2018-06-20T00:00:00.000Z,2018-06-21T19:04:47.000Z,beleza_saude,46.0,1290.0,1.0,950.0,25.0,12.0,19.0,health_beauty,921c8a95cd7f1ba09d26223716253455,13070,campinas,SP,-27.0,On Time
d0028facea13f508e880202d7097a5a1,d2509c13692836fc0449e88cf9eb4858,delivered,2018-04-20T12:57:23.000Z,2018-04-25T03:51:13.000Z,2018-04-25T15:25:00.000Z,2018-04-27T12:08:59.000Z,2018-05-09T00:00:00.000Z,1.0,8cefe1c6f2304e7e6825150218ffc58c,ea8482cd71df3c1969d7b9473ff13abc,2018-05-02T03:51:13.000Z,27.99,7.39,1.0,boleto,1.0,35.38,ffc1a4080d67b1d3d06749f4f2d3ee59,4.0,null,null,2018-04-28T00:00:00.000Z,2018-05-01T20:41:51.000Z,telefonia,59.0,818.0,6.0,300.0,17.0,4.0,12.0,telephony,00050ab1314c0e55a6ca13cf7181fecf,13084,campinas,SP,-12.0,On Time
812114e24682287b1f3eb86f112707de,de94552cb07b7cbb5219d1616580a65c,delivered,2017-12-04T21:36:35.000Z,2017-12-05T09:30:45.000Z,2017-12-07T15:19:17.000Z,2017-12-19T15:53:56.000Z,2017-12-28T00:00:00.000Z,1.0,06bf70b6e1d67d96308235ef350edc61,2c9e548be18521d1c43cde1c582c6de8,2017-12-11T08:30:50.000Z,79.9,15.32,1.0,boleto,1.0,95.22,7a3fc44fbc09b8f38a5bead35c987785,4.0,null,null,2017-12-20T00:00:00.000Z,2017-12-20T23:27:27.000Z,brinquedos,48.0,284.0,2.0,950.0,105.0,10.0,11.0,toys,3526e8083b82706f0a66e58c84044cab,32676,betim,MG,-9.0,On Time
bc6c1d04178771a3ac9547f4770ab5f0,8fd50d32ac51d30d03b416bf7b27e72d,delivered,2017-10-27T10:27:03.000Z,2017-10-27T10:35:22.000Z,2017-10-31T21:53:55.000Z,2017-11-09T21:41:44.000Z,2017-11-17T00:00:00.000Z,1.0,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2017-11-03T10:35:22.000Z,59.9,17.67,1.0,credit_card,4.0,77.57,c123f6ddcc806bd9bed231d6163f267e,5.0,null,null,2017-11-10T00:00:00.000Z,2017-11-13T10:01:10.000Z,ferramentas_jardim,56.0,348.0,2.0,1550.0,30.0,22.0,30.0,garden_tools,8cf98f

# Generate Executive KPIs

In [0]:
total_orders = df["order_id"].nunique()
gmv = df["payment_value"].sum()
aov = gmv / total_orders
avg_review = df["review_score"].mean()

print("Total Orders:", total_orders)
print("GMV:", round(gmv,2))
print("AOV:", round(aov,2))
print("Avg Review:", round(avg_review,2))

Total Orders: 99441
GMV: 20579664.01
AOV: 206.95
Avg Review: 4.02


# Analysis 1 — Delivery Delay vs Customer Satisfaction

In [0]:
df.groupby("delivery_status")["review_score"].mean()

delivery_status
Delayed    2.253393
On Time    4.132788
Name: review_score, dtype: float64

# Analysis 2 — Top Revenue Categories

In [0]:
category_perf = df.groupby(
    "product_category_name_english"
).agg({
    "payment_value":"sum",
    "review_score":"mean",
    "order_id":"nunique"
}).sort_values(
    by="payment_value",
    ascending=False
).head(10)

category_perf

,payment_value,review_score,order_id
product_category_name_english,,,
bed_bath_table,1743998.80,3.890605,9417
health_beauty,1662963.59,4.137026,8836
computers_accessories,1599481.06,3.936089,6689
furniture_decor,1443963.61,3.912158,6449
watches_gifts,1430553.48,4.017692,5624
sports_leisure,1400223.07,4.107470,7720
housewares,1097900.09,4.060428,5884
auto,855095.68,4.064279,3897
garden_tools,840721.59,4.023914,3518


# Analysis 3 — Repeat Customer Analysis

In [0]:
customer_orders = df.groupby(
    "customer_unique_id"
)["order_id"].nunique().reset_index()

customer_orders.columns = [
    "customer_unique_id",
    "order_count"
]

In [0]:
repeat_rate = (
    customer_orders["order_count"] > 1
).mean()

print("Repeat Purchase Rate:", round(repeat_rate*100,2), "%")

Repeat Purchase Rate: 3.12 %
